# Figure 2 and its supporting-information counterparts

Band-passed sea surface height and surface speed on four forecast lead days of one
cycle, with the detected eddy contours on top: the mapped altimetry in the upper
row and the forecast in the lower, as published. One figure per region and field.

The two products are on their own grids and are stored in separate files, so no
interpolation is applied here that the figure did not make.

Eddy contours are a CF contiguous ragged array: the vertices of eddy `i` run from
`contour_vertex_start[i]` for `contour_vertex_count[i]` points.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    PROJ, HAVE_CARTOPY = ccrs.PlateCarree(), True
except ImportError:
    PROJ, HAVE_CARTOPY = None, False
    print("cartopy is not installed -- drawing the maps without coastlines")

# Cartopy fetches its coastline data from the network the first time it is used.
# Try once here, so a machine that is offline or has no cached copy degrades to a
# map without coastlines instead of failing in the middle of a figure.
HAVE_COAST = False
if HAVE_CARTOPY:
    try:
        next(iter(cfeature.COASTLINE.geometries()))
        HAVE_COAST = True
    except Exception as exc:
        print(f"cartopy cannot reach its coastline data ({type(exc).__name__}) -- "
              f"drawing the maps without coastlines")

TKW = dict(transform=PROJ) if HAVE_CARTOPY else {}


def map_features(ax, labels=True, land="lightgray", lw=0.4, zland=3, zcoast=4):
    """Coastlines and gridlines where available; plain axes otherwise."""
    if HAVE_COAST:
        ax.add_feature(cfeature.LAND, facecolor=land, zorder=zland)
        ax.add_feature(cfeature.COASTLINE, linewidth=lw, zorder=zcoast)
    if HAVE_CARTOPY:
        gl = ax.gridlines(draw_labels=labels, linestyle="--", lw=0.2)
        gl.top_labels = gl.right_labels = False
        return gl
    ax.tick_params(labelsize=7)
    return None


FIELDS = [("ssh",   dict(vmin=-0.2, vmax=0.2, cmap="RdYlBu_r",
                         label="SSH (m)")),
          ("speed", dict(vmin=0.0,  vmax=0.5, cmap="YlOrRd",
                         label="Speed (m s$^{-1}$)"))]
ROW_ORDER = ["observed", "forecast"]
ROW_LABEL = {"observed": "Altimetry", "forecast": "Forecast"}


def draw_contours(ax, ds, day):
    sel = np.flatnonzero(ds.eddy_lead_day.values == day)
    for i in sel:
        s = int(ds.contour_vertex_start.values[i])
        n = int(ds.contour_vertex_count.values[i])
        ax.plot(ds.contour_lon.values[s:s + n], ds.contour_lat.values[s:s + n],
                "r-" if ds.eddy_polarity.values[i] == 1 else "b-",
                lw=1.0, zorder=5, **TKW)
    return len(sel)


# group the files by region, then by product
groups = {}
for region, path in by_region(need("Figure2_*.nc")):
    with xr.open_dataset(path) as ds:
        groups.setdefault(region, {})[ds.attrs["product"]] = path

for region, byprod in groups.items():
    rows = [p for p in ROW_ORDER if p in byprod]
    if len(rows) < 2:
        print(f"{region}: only {rows} in this deposit")
    opened = {p: xr.open_dataset(byprod[p]) for p in rows}
    days = list(opened[rows[0]].lead_day.values)

    for var, cfg in FIELDS:
        fig, axes = plt.subplots(len(rows), len(days),
                                 figsize=(3.4 * len(days), 2.9 * len(rows)),
                                 dpi=130, squeeze=False,
                                 subplot_kw={"projection": PROJ}
                                 if HAVE_CARTOPY else None)
        pcm = None
        for r, prod in enumerate(rows):
            ds = opened[prod]
            for c, day in enumerate(days):
                ax = axes[r][c]
                pcm = ax.pcolormesh(ds.lon, ds.lat, ds[var].sel(lead_day=day),
                                    cmap=cfg["cmap"], vmin=cfg["vmin"],
                                    vmax=cfg["vmax"], shading="auto",
                                    zorder=1, **TKW)
                n = draw_contours(ax, ds, int(day))
                map_features(ax, labels=(c == 0))
                if r == 0:
                    ax.set_title(f"Lead day {int(day)}", fontsize=10,
                                 fontweight="bold")
                if c == 0:
                    ax.text(-0.12, 0.5, ROW_LABEL[prod], transform=ax.transAxes,
                            rotation=90, va="center", ha="center", fontsize=10,
                            fontweight="bold")
                ax.text(0.02, 0.95, f"({chr(97 + r * len(days) + c)}) {n} eddies",
                        transform=ax.transAxes, fontsize=6.5, va="top",
                        zorder=12,
                        bbox=dict(fc="white", alpha=0.8, pad=0.8, ec="none"))
        cb = fig.colorbar(pcm, ax=axes.ravel().tolist(), orientation="vertical",
                          fraction=0.02, pad=0.01, shrink=0.85)
        cb.set_label(cfg["label"], fontsize=9)
        fig.suptitle(f"{region}: cycle "
                     f"{opened[rows[0]].attrs['forecast_cycle']}",
                     fontsize=11, fontweight="bold")
        out = (FIGDIR /
               f"Figure2_{var}_{region.replace(' ', '_')}.png")
        fig.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("wrote", out)

    lic = [d.attrs["licence"] for d in opened.values() if "licence" in d.attrs]
    if lic:
        print("\n" + lic[0])
    for d in opened.values():
        d.close()
